# Aim 2 DL arm: Olsen 2020 BiGRU training (Colab)

Trains `OlsenBiGRU` (`thesis_pipeline/dl_models.py`) on packed stage-B arrays with
`scripts/train_aim2_dl.py`. Design: vault pre-registration note plus the build design
(sections 1, 2.3-2.4, 4, 5).

**Configs:** M = [RR, EDR] (faithful Olsen), P4 = [RR, EDR, THOR, ABDO]. F6 is deferred
(Aim 1 airflow picker defect). **Seeds:** 42, 43, 44 (model seed; the split seed stays 42).

**Test protection (hard rules):**
* Drive holds **train and val only** (`MyDrive/Thesis/data/dl_cache_v1/model_v1/{train,val}`).
  This notebook asserts there is no `test` folder and never copies one.
* No test inference here. Test predictions are made once, after the code freeze, on the Mac
  (`scripts/predict_aim2_dl.py --split test --freeze-tag ...`), and metrics only by
  `scripts/evaluate_aim2_dl.py`.
* Pilot / smoke numbers are for debugging and are never reported.

**Order:** cells 1-5 every session; cell 6 (gate) once, which locks the recipe for all
configs and seeds; cell 7 trains one seed with auto-resume (re-run after a disconnect);
cell 8 runs three seeds concurrently; cell 9 summarises, locks the aggregator from M seed 42
and re-finalises (no retraining) any run whose aggregator differs from the lock.

Outputs are written to VM-local disk (`/content/out`, `/content/ckpt`) and mirrored to Drive
(`RESULTS/<config>/seed<k>`). A Drive I/O error is logged and retried on the next save; it
does not stop training.

In [ ]:
# 1. Mount Drive and fix paths
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
import os, json, time, hashlib, shutil, subprocess, pathlib

DRIVE_ROOT = '/content/drive/MyDrive/Thesis'
DATA_DRIVE = f'{DRIVE_ROOT}/data/dl_cache_v1/model_v1'      # train/ and val/ ONLY
RESULTS = f'{DRIVE_ROOT}/results/aim2_dl_olsen_v1'
LOGS = f'{DRIVE_ROOT}/logs/aim2_dl_olsen_v1'
LOCAL = '/content/dl_cache/model_v1'
CKPT_LOCAL = '/content/ckpt'
OUT_LOCAL = '/content/out'                           # local outputs, mirrored to RESULTS
for d in (RESULTS, LOGS):
    os.makedirs(d, exist_ok=True)
assert os.path.isdir(f'{DATA_DRIVE}/train') and os.path.isdir(f'{DATA_DRIVE}/val'), 'upload train/ and val/ first'
assert not os.path.exists(f'{DATA_DRIVE}/test'), 'TEST DATA MUST NEVER BE ON DRIVE: remove it before continuing'
print('Drive OK:', DATA_DRIVE, '->', RESULTS)

In [ ]:
# 2. Clone / update the repo (same PAT pattern as sleep_filter_recovery_colab.ipynb)
from google.colab import userdata
PAT = userdata.get('GITHUB_PAT_THESIS')
REPO = 'jameswudo1019hack/thesis-bme-pipeline'
if pathlib.Path('/content/Code').exists():
    subprocess.run(['git', '-C', '/content/Code', 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', '--depth=50', f'https://{PAT}@github.com/{REPO}.git', '/content/Code'], check=True)
for f in ('thesis_pipeline/dl_train.py', 'thesis_pipeline/dl_models.py', 'scripts/train_aim2_dl.py',
          'scripts/bench_bigru.py', 'splits/aim2_seed42.json'):
    assert pathlib.Path('/content/Code', f).exists(), f'{f} missing: push the DL build first'
COMMIT = subprocess.run(['git', '-C', '/content/Code', 'rev-parse', 'HEAD'], capture_output=True, text=True).stdout.strip()
print('HEAD', COMMIT)

In [ ]:
# 3. Dependencies (torch is preinstalled on Colab)
%cd /content/Code
!pip install -q neurokit2 pyarrow click scikit-learn lxml mne 2>&1 | tail -2
!pip install -q -e . 2>&1 | tail -2
import torch
print('torch', torch.__version__, 'cuda', torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

In [ ]:
# 4. Run parameters
CONFIG = 'M'                 # 'M' or 'P4' (F6 deferred)
SEEDS = [42, 43, 44]
CHANNELS = {'M': ['RR', 'EDR'], 'P4': ['RR', 'EDR', 'THOR', 'ABDO']}[CONFIG]
RECIPE_LOCK = f'{RESULTS}/RECIPE_LOCK.json'          # written once by the gate (cell 6)
AGG_LOCK = f'{RESULTS}/AGGREGATOR_LOCK.json'         # written once from M seed 42 (cell 9)
RECIPE = json.load(open(RECIPE_LOCK))['recipe'] if os.path.exists(RECIPE_LOCK) else None
print('config', CONFIG, CHANNELS, 'recipe', RECIPE)

In [ ]:
# 5. Copy the config's train/val shards to local disk and verify sha256 against MANIFEST.json
def sha256(p, buf=1 << 22):
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        for b in iter(lambda: f.read(buf), b''):
            h.update(b)
    return h.hexdigest()

split_sha = sha256('/content/Code/splits/aim2_seed42.json')
t0 = time.time()
for split in ('train', 'val'):
    src, dst = f'{DATA_DRIVE}/{split}', f'{LOCAL}/{split}'
    os.makedirs(dst, exist_ok=True)
    man = json.load(open(f'{src}/MANIFEST.json'))
    assert man['split'] == split, f'{src} manifest says {man["split"]!r}'
    assert man['split_sha256'] == split_sha, 'cache packed from a different split file'
    names = [n for n in man['sha256'] if not n.startswith('signals_') or any(f'_{c}_' in n for c in CHANNELS)]
    for n in names:
        s, d = f'{src}/{n}', f'{dst}/{n}'
        if not (os.path.exists(d) and os.path.getsize(d) == os.path.getsize(s) and sha256(d) == man['sha256'][n]):
            shutil.copyfile(s, d)
            assert sha256(d) == man['sha256'][n], f'sha256 mismatch after copy: {n}'
    shutil.copyfile(f'{src}/MANIFEST.json', f'{dst}/MANIFEST.json')
    print(split, len(names), 'files verified;', man['n_subjects'], 'subjects')
assert not os.path.exists(f'{LOCAL}/test')
print(f'copy + verify {time.time() - t0:.0f} s')

In [ ]:
# 6. Benchmark gate (ONCE, before any production run). Locks the recipe for all configs and seeds.
#    R-A: batch-128 single >= 800 win/s or 3 processes >= 1500 total; R-B: 400-800; R-C: < 400 or T4.
import sys
sys.path.insert(0, '/content/Code/scripts')
from bench_bigru import gate_recipe
if os.path.exists(RECIPE_LOCK):
    print('recipe already locked:', json.load(open(RECIPE_LOCK)))
else:
    cin = len(CHANNELS)
    !nvidia-smi --query-gpu=name,memory.total --format=csv
    !python scripts/bench_bigru.py --device cuda --amp --cin {cin} --batch 128 --batch 256 --batch 512 --json-out /content/gate_single.json
    !python scripts/bench_bigru.py --device cuda --amp --cin {cin} --batch 128 --procs 3 --json-out /content/gate_multi.json
    !python scripts/bench_bigru.py --device cuda --amp --cin {cin} --batch 128 --cache-dir {LOCAL}/train --json-out /content/gate_mmap.json
    single = json.load(open('/content/gate_single.json'))
    multi = json.load(open('/content/gate_multi.json'))[0]
    for f in ('/content/gate_single.json', '/content/gate_multi.json', '/content/gate_mmap.json'):
        assert os.path.exists(f), f'{f} missing: that benchmark failed (see its output above); nothing locked'
    mmap = json.load(open('/content/gate_mmap.json'))[0]
    s128 = next(r['train_windows_per_sec'] for r in single if r['batch'] == 128)
    gpu = single[0]['gpu'] or ''
    RECIPE = gate_recipe(s128, multi['total_train_windows_per_sec'], t4='T4' in gpu)
    units = input('Compute-unit balance now (from the Colab resources panel): ')
    lock = {'recipe': RECIPE, 'gpu': gpu, 'single_128': s128, 'multi_total': multi['total_train_windows_per_sec'],
            'mmap_gather_128': mmap['train_windows_per_sec'], 'single': single, 'commit': COMMIT,
            'compute_units_at_gate': units, 'locked': time.strftime('%Y-%m-%dT%H:%M:%S')}
    json.dump(lock, open(RECIPE_LOCK, 'w'), indent=2)
    print('LOCKED recipe', RECIPE, lock)

In [ ]:
# 7. Train ONE seed with auto-resume (checkpoints and outputs mirrored to Drive every pass). Re-run after a disconnect.
assert RECIPE, 'run the gate (cell 6) first'
SEED = 42
run = f'{RESULTS}/{CONFIG}/seed{SEED}'
os.makedirs(run, exist_ok=True)
if CONFIG == 'M' and SEED == 42:
    agg = 'auto'                                   # the pre-declared rule runs on M seed-42 validation
elif os.path.exists(AGG_LOCK):
    agg = json.load(open(AGG_LOCK))['aggregator']  # frozen choice for every other run
else:
    agg = 'auto'
    print('NOTE: aggregator not locked yet; cell 9 re-finalises this run with the locked aggregator (no retraining)')
cmd = (f'python scripts/train_aim2_dl.py --config {CONFIG} --model-seed {SEED} --recipe {RECIPE} '
       f'--train-dir {LOCAL}/train --val-dir {LOCAL}/val --ckpt-dir {CKPT_LOCAL}/{CONFIG}/seed{SEED} '
       f'--mirror-dir {run} --out-dir {OUT_LOCAL}/{CONFIG}/seed{SEED} --aggregator {agg}')
!{cmd} 2>&1 | tee -a {LOGS}/train_{CONFIG}_seed{SEED}.log

In [ ]:
# 8. OPTIONAL: three seeds concurrently on one GPU (only if the gate showed >= 1.5x aggregate throughput).
#    Each process resumes from its own Drive mirror; logs go to Drive. Re-run to resume after a disconnect.
#    A process that exits with an error is restarted (it resumes) at most MAX_RESTARTS times.
assert RECIPE
MAX_RESTARTS = 2

def launch(seed):
    run = f'{RESULTS}/{CONFIG}/seed{seed}'
    os.makedirs(run, exist_ok=True)
    agg = 'auto' if (CONFIG == 'M' and seed == 42) else (
        json.load(open(AGG_LOCK))['aggregator'] if os.path.exists(AGG_LOCK) else 'auto')
    if agg == 'auto' and not (CONFIG == 'M' and seed == 42):
        print(f'NOTE: seed {seed}: aggregator not locked yet; cell 9 re-finalises it with the lock (no retraining)')
    cmd = ['python', 'scripts/train_aim2_dl.py', '--config', CONFIG, '--model-seed', str(seed), '--recipe', RECIPE,
           '--train-dir', f'{LOCAL}/train', '--val-dir', f'{LOCAL}/val',
           '--ckpt-dir', f'{CKPT_LOCAL}/{CONFIG}/seed{seed}', '--mirror-dir', run,
           '--out-dir', f'{OUT_LOCAL}/{CONFIG}/seed{seed}', '--aggregator', agg]
    log = open(f'{LOGS}/train_{CONFIG}_seed{seed}.log', 'a')
    return subprocess.Popen(cmd, cwd='/content/Code', stdout=log, stderr=subprocess.STDOUT)

procs = {seed: launch(seed) for seed in SEEDS}
restarts = {seed: 0 for seed in SEEDS}
while any(p.poll() is None for p in procs.values()):
    time.sleep(300)
    for seed in SEEDS:
        rc = procs[seed].poll()
        if rc not in (None, 0) and restarts[seed] < MAX_RESTARTS:
            restarts[seed] += 1
            print(time.strftime('%H:%M'), seed, f'exited {rc}; restart {restarts[seed]}/{MAX_RESTARTS} (resumes)')
            procs[seed] = launch(seed)
        h = f'{RESULTS}/{CONFIG}/seed{seed}/history.csv'
        tail = open(h).read().strip().splitlines()[-1] if os.path.exists(h) else 'no pass yet'
        print(time.strftime('%H:%M'), seed, tail[:160])
print({s: p.returncode for s, p in procs.items()}, 'restarts', restarts)
if any(p.returncode for p in procs.values()):
    print('WARNING: a seed still failed; read its log in', LOGS, 'and re-run this cell to resume')

In [ ]:
# 9. Summary; lock the aggregator from M seed 42 (pre-declared rule, validation only); re-finalise mismatches
import pandas as pd
for cfg in ('M', 'P4'):
    for seed in SEEDS:
        run = f'{RESULTS}/{cfg}/seed{seed}'
        if os.path.exists(f'{run}/metrics.json'):
            m = json.load(open(f'{run}/metrics.json'))
            print(cfg, seed, 'val AUC', round(m['best_val_auc_mean_agg'], 4), 'best pass', m['best_pass'],
                  'passes', m['passes_run'], m['stop_reason'], 'agg', m['aggregator'], 'thr', m['threshold'])
        elif os.path.exists(f'{run}/history.csv'):
            print(cfg, seed, 'running:', len(pd.read_csv(f'{run}/history.csv')), 'passes')
m42 = f'{RESULTS}/M/seed42/postproc.json'
if os.path.exists(m42) and not os.path.exists(AGG_LOCK):
    pp = json.load(open(m42))
    json.dump({'aggregator': pp['aggregator'], 'rule': pp['aggregator_rule'], 'from': m42,
               'locked': time.strftime('%Y-%m-%dT%H:%M:%S')}, open(AGG_LOCK, 'w'), indent=2)
    print('LOCKED aggregator', pp['aggregator'], pp['aggregator_rule']['aucs'])
# Every finished run must use the locked aggregator. A run that finalised with 'auto' before the lock
# existed is re-finalised with the locked choice: the trainer resumes with done=True and only rewrites
# val_predictions / postproc / metrics (aggregator is not part of the config hash).
if os.path.exists(AGG_LOCK):
    LOCKED = json.load(open(AGG_LOCK))['aggregator']
    for cfg in ('M', 'P4'):
        if cfg != CONFIG:
            print(f'{cfg}: switch CONFIG to {cfg} and re-run cells 4-5 + this cell to check it'); continue
        for seed in SEEDS:
            run = f'{RESULTS}/{cfg}/seed{seed}'
            if not os.path.exists(f'{run}/postproc.json'):
                continue
            got = json.load(open(f'{run}/postproc.json'))['aggregator']
            if got == LOCKED:
                continue
            print(f'{cfg} seed {seed}: aggregator {got} != locked {LOCKED}; re-finalising (no retraining)')
            cmd = (f'python scripts/train_aim2_dl.py --config {cfg} --model-seed {seed} --recipe {RECIPE} '
                   f'--train-dir {LOCAL}/train --val-dir {LOCAL}/val --ckpt-dir {CKPT_LOCAL}/{cfg}/seed{seed} '
                   f'--mirror-dir {run} --out-dir {OUT_LOCAL}/{cfg}/seed{seed} --aggregator {LOCKED}')
            !{cmd} 2>&1 | tee -a {LOGS}/train_{cfg}_seed{seed}.log
            got = json.load(open(f'{run}/postproc.json'))['aggregator']
            assert got == LOCKED, f'{cfg} seed {seed} still has aggregator {got}'
    print('all finished runs of', CONFIG, 'use the locked aggregator', LOCKED)

**Next steps (on the Mac, not here):** after the training freeze, tag the code, recompute
validation predictions on the Mac (`predict_aim2_dl.py --split val --refit-postproc
--aggregator-lock AGGREGATOR_LOCK.json`), then run test inference once per (config, seed) with
`--split test --freeze-tag <tag> --prereg <note> --aggregator-lock AGGREGATOR_LOCK.json
--postproc postproc_mps.json` (predict refuses a postproc from another checkpoint, device or
aggregator) and evaluate with `evaluate_aim2_dl.py --split test --prereg <note>`.